# Phase 3 — Evidence Validation

**Project:** VIGILOX Document Intelligence
**Phase Status:** Complete ✅

## 1. Purpose

Phase 3 was introduced to verify that structured fields produced by the LLM are actually supported by OCR evidence.

Phase 2 could return valid structured JSON, but schema validity alone does not prove that the extracted value is correct.

The Phase 3 goal was therefore:

```text
Structured Extraction
        ↓
Evidence Validation
        ↓
Trusted / Reviewable Output
```

The validator checks both the existence of evidence and whether that evidence semantically supports the extracted field.

---

## 2. Structural Evidence Validation

The first validation layer checks the structure of evidence references.

For every extracted non-null field, the system verifies that:

* `source_line_ids` are provided
* Each referenced line ID exists in the OCR input
* Invalid or invented references are rejected

Example:

```text
Available OCR IDs:
L0 ... L7

Returned evidence:
L45
```

Result:

```text
INVALID_SOURCE_LINE_ID
```

The validator does not attempt to guess that `L45` might mean `L4` and `L5`.

This is an important reliability rule:

> Invalid evidence is flagged, not automatically repaired.

---

## 3. Source Line ID Validation

OCR evidence uses explicit string identifiers:

```text
L0
L1
L2
L3
...
```

This design makes each extracted field traceable to a specific OCR line.

Example:

```text
L4 → EXPIRES
L5 → 24 MAR 2021
```

A correct expiry extraction can reference:

```text
L4, L5
```

This allows deterministic verification of the evidence supplied by the LLM.

---

## 4. Semantic Evidence Validation

Structural validity is not enough.

A valid source line may exist but still have no relationship to the extracted field.

Example:

```text
full_name = M.GREEN
source = L0

L0 = 1099 4265 1706 9065
```

The line ID exists, but the evidence does not support the name.

The validator therefore produces an evidence mismatch.

Conceptually:

```text
Valid Line ID
     ↓
Retrieve OCR Text
     ↓
Compare Against Extracted Value
     ↓
Match / Mismatch
```

This became the second validation layer.

---

## 5. Evidence Mismatch Detection

For normal text fields, values are normalized before comparison.

Normalization helps avoid unnecessary mismatches caused by differences such as:

* Capitalization
* Spaces
* Punctuation
* Minor formatting differences

For example:

```text
M.GREEN
M GREEN
m.green
```

can be compared in a normalized form.

If the normalized extracted value cannot be found in the referenced OCR evidence, the field is flagged as:

```text
EVIDENCE_MISMATCH
```

This prevents an LLM from attaching an otherwise valid but unrelated OCR line to a field.

---

## 6. Date Semantic Validation

Dates require more than normal string matching.

For example:

```text
Structured value:
2021-03-24

OCR evidence:
24 MAR 2021
```

These strings are different, but they represent the same date.

Phase 3 therefore added date-aware semantic comparison.

Supported forms include patterns such as:

```text
2021-03-24
24 MAR 2021
24 March 2021
24/03/2021
24/03/21
24-03-2021
```

The validator converts supported date formats into comparable date representations before checking whether the evidence supports the extracted value.

---

## 7. Context Validation for Date Fields

A date value may be correct but still be assigned to the wrong field.

Example:

```text
24 MAR 2021
```

alone does not prove whether the date means:

* Expiry date
* Issue date
* Date of birth

Therefore date fields also require semantic context where appropriate.

For example:

```text
L4 → EXPIRES
L5 → 24 MAR 2021
```

supports:

```text
expiry_date = 2021-03-24
```

because both the value and the `EXPIRES` context are present.

If only:

```text
L5 → 24 MAR 2021
```

is referenced, the date may match, but the expiry context is missing.

The validator can therefore flag:

```text
CONTEXT_MISSING
```

---

## 8. Context Rules Completed

Phase 3 currently validates contextual labels for the main date fields.

### Expiry Date

Examples of accepted context include:

```text
EXPIRES
EXPIRY
EXPIRY DATE
EXPIRATION
VALID UNTIL
VALID TO
```

### Date of Birth

Examples include:

```text
DOB
DATE OF BIRTH
BIRTH DATE
```

### Issue Date

Examples include:

```text
ISSUED
ISSUE DATE
DATE ISSUED
PRINT DATE
```

This reduces the risk of assigning an otherwise valid date to the wrong semantic field.

---

## 9. Independent Semantic Validation Test

The evidence validator was tested independently from Groq using known OCR lines and a known structured extraction.

The test result was:

```text
========== V2 TEST ==========

All semantic evidence is valid.
```

This was important because it confirmed that the validator itself worked correctly without depending on LLM behavior.

---

## 10. Integrated Phase 3 Result

The complete SIA pipeline produced:

```text
L0 → 1099 4265 1706 9065
L3 → Security Industry Authority
L4 → EXPIRES
L5 → 24 MAR 2021
L6 → M.GREEN
```

The structured extraction returned:

```text
Document Type:
sia_badge

Full Name:
M.GREEN
Evidence: L6

Licence Number:
1099 4265 1706 9065
Evidence: L0

Expiry Date:
2021-03-24
Evidence: L4, L5

Issuer:
Security Industry Authority
Evidence: L3
```

The final validation result was:

```text
All evidence references are valid.
```

This confirmed that the integrated:

```text
OCR
→ Groq
→ Pydantic
→ Evidence Validation
```

pipeline was working successfully.

---

## 11. Important Engineering Findings

### Schema validation is not evidence validation

Pydantic can confirm that an LLM response has the correct structure, but it cannot prove that the extracted values are supported by the document.

Therefore both layers are required.

---

### Evidence must remain traceable

Every important extracted field should retain a direct reference to the OCR line or lines that support it.

This makes extraction:

* Auditable
* Debuggable
* Reviewable
* Suitable for later human verification

---

### Invalid evidence should never be guessed

If an LLM returns an invalid reference such as:

```text
L45
```

the system should flag it rather than interpreting it as:

```text
L4 + L5
```

Automatic guessing would weaken auditability and could hide extraction errors.

---

### Semantic matching must be field-aware

A generic text comparison is insufficient for all fields.

Dates need:

* Format normalization
* Semantic comparison
* Context validation

This shows that trustworthy document extraction requires deterministic domain-aware checks after the LLM.

---

### LLM output should be treated as a proposal, not ground truth

The LLM performs interpretation, but the deterministic validator checks whether that interpretation is supported by OCR evidence.

The resulting architecture is therefore:

```text
LLM proposes
      ↓
Validator verifies
```

rather than:

```text
LLM returns
      ↓
Automatically trust
```

---

## 12. Phase 3 Final Conclusion

Phase 3 successfully added a deterministic evidence-validation layer to VIGILOX Document Intelligence.

The completed validation process now checks:

```text
Evidence supplied?
      ↓
Source line exists?
      ↓
Referenced text supports value?
      ↓
Date formats semantically match?
      ↓
Required field context exists?
```

This significantly improves the reliability and auditability of the structured extraction pipeline.

The final Phase 3 architecture is:

```text
OCR Output
    ↓
Structured LLM Extraction
    ↓
Pydantic Validation
    ↓
Structural Evidence Validation
    ↓
Semantic Evidence Validation
    ↓
Validated Structured Data
```

**Phase 3 is complete.**
